## Grant Deduplication (S2)

Deduplicates newly-queried Dimensions grants (from S1) against tracked/historical grant sources - the last report's curated data and the GFI Grants Tracker (Airtable export) - before the survivors go to S3 for LLM scope screening.

This is a manual notebook, not an automated script: four steps pause for you to review and confirm full/partial title matches in an exported CSV before continuing. Redesigned from `SHEEP/Funding/grant_deduplication.ipynb` - see that notebook for the original prototype and its "Stella" validation notes.

Note: this notebook does not compare anything against a "ground truth" dataset. There isn't one - a new grant's scope/pillar labels come from combined LLM (S3) and human review, which is what makes them trustworthy going forward. What this notebook does is purely deduplication: telling apart grants that are already tracked (in the last report or the Grants Tracker) from genuinely new ones that need to go through S3.

**Before running:** set `RUN_TABLE` in the config cell below to the run you want to process - the name `pipeline_funding.py` printed when it asked you to run this notebook (e.g. `run_260721_1400`).

In [1]:
import sys
from pathlib import Path
from datetime import datetime

import duckdb
import pandas as pd
import numpy as np
import pycountry
import re

sys.path.append(str(Path.cwd()))
from Funding_dedup_helpers import (
    is_empty, is_zero, normalize_title, assign_stable_row_id,
    gap_fill, gap_fill_researchers_and_orgs,
    export_highlighted_diff, export_for_review, apply_reviewed_decisions,
)

# CONFIG - edit before running
#RUN_TABLE = 'run_YYMMDD_HHMM'  # <-- set this to match the run printed by pipeline_funding.py
RUN_TABLE = 'run_260727_0922'
RUN_TIMESTAMP = '260727_0922'  # for audit/review filenames, so reruns don't overwrite

DB_PATH = Path('funding.db')       # self-contained in Pipeline/Funding, mirrors Publications
RAW_DATA_DIR = Path('raw_data')    # self-contained in Pipeline/Funding
REVIEW_DIR = Path('data_review')   # review CSVs (export_for_review / apply_reviewed_decisions)
AUDIT_DIR = Path('data_audit')     # highlighted-diff audit exports + final table Excel mirrors
REVIEW_DIR.mkdir(exist_ok=True)
AUDIT_DIR.mkdir(exist_ok=True)

LAST_REPORT_FILE = 'Funding2026_inscope.xlsx'
GRANTS_TRACKER_FILE = 'GrantsTracker_2026-06-30_edit-start-year.xlsx'  # update to the latest tracker export as needed

# Whether last_report_data (LAST_REPORT_FILE) has global or Europe-only historical coverage.
# 'europe': last_report_data only has Europe grants (true through the 2026 report) - Grants
#   Tracker filtering below applies this cycle's date window to Europe grants only, and does a
#   one-time full-history backfill of non-Europe grants (never tracked before, so there's no
#   prior-cycle coverage to rely on for them).
# 'global': last_report_data already has full worldwide coverage (true from whichever cycle
#   first completes the non-Europe backfill onward) - just the incremental date window applies,
#   no region carve-out needed. Flip this to 'global' once that's confirmed true.
LAST_REPORT_SCOPE = 'europe'

# Grants Tracker date filter - only rows with 'EXT_Date added' in this window are included.
GT_DATE_ADDED_START = '2026-01-01'
GT_DATE_ADDED_END   = '2026-12-31'

FUZZY_THRESHOLD = 85  # 0-100; rapidfuzz token_sort_ratio threshold for partial title matches

# Funding start-year filter - applied to BOTH last_report_data and grants_tracker_data (not just
# new Dimensions queries, which use their own START_YEAR/END_YEAR in S1). Rows with a missing/blank
# start-year field are EXCLUDED, not assumed in-range - see the printed NOTE lines in the load/filter
# cells below for how many rows that affects each run.
FUNDING_START_YEAR_MIN = 2020
FUNDING_START_YEAR_MAX = 2025

DEDUP_TABLE = f'{RUN_TABLE}_dedup'
RUN_DATE = datetime.today().strftime('%y%m%d')


print(f"RUN_TABLE = {RUN_TABLE}")
print(f"DEDUP_TABLE (output, S3 reads this) = {DEDUP_TABLE}")

RUN_TABLE = run_260727_0922
DEDUP_TABLE (output, S3 reads this) = run_260727_0922_dedup


### 1. Last report's curated data

The base dataset this run gap-fills and extends. Already-tracked grants here are not re-scored - they keep whatever scope/pillar labels they were previously assigned (via LLM + human review in an earlier cycle).

In [2]:
def clean_dtypes(df):
    df = df.copy()
    datetime_cols = [
        'Date request submitted', 'Date award announced',
        'Project start date', 'Date added', 'Last modified',
    ]
    for col in datetime_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce').astype('datetime64[us]')

    year_cols = ['Year request submitted', 'Year project started', 'End date']
    for col in year_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')

    def _clean_months(x):
        if pd.isna(x) or isinstance(x, pd.Timestamp):
            return None
        try:
            return int(float(x))
        except (ValueError, TypeError):
            return None

    if 'Duration of award (months)' in df.columns:
        df['Duration of award (months)'] = df['Duration of award (months)'].apply(_clean_months).astype('Int64')

    if 'duration (years)' in df.columns:
        def _clean_years(x):
            if pd.isna(x) or isinstance(x, pd.Timestamp):
                return None
            try:
                val = int(float(x))
                return val if 0 <= val <= 20 else None
            except (ValueError, TypeError):
                return None
        df['duration (years)'] = df['duration (years)'].apply(_clean_years).astype('Int64')

    return df


last_report_data = pd.read_excel(RAW_DATA_DIR / LAST_REPORT_FILE)
if 'Production platform' in last_report_data.columns:
    last_report_data = last_report_data.rename(columns={'Production platform': 'AP pillar'})
# 'Unnamed: 15' is a genuinely blank column in the Airtable export (0 non-null values) -
# repurposed to preserve whatever was in 'Identification code' before a confirmed Dimensions
# title match overwrites it with the Dimensions grant ID (see REVIEW POINT 3 below).
if 'Unnamed: 15' in last_report_data.columns:
    last_report_data = last_report_data.rename(columns={'Unnamed: 15': 'Legacy identification code'})
    # All-blank Excel columns load as float64 (NaN is a float) - cast to object so it can
    # hold the string IDs REVIEW POINT 3 stashes into it later without a dtype error.
    last_report_data['Legacy identification code'] = last_report_data['Legacy identification code'].astype(object)
last_report_data = clean_dtypes(last_report_data)
last_report_data = assign_stable_row_id(last_report_data, 'lrd_row_id')

n_loaded = len(last_report_data)

# Only grants with a funding start year in [FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX] are in scope.
# NOTE: rows with a missing/blank 'Year project started' are EXCLUDED here - a blank year can't be
# confirmed to fall in range, so it's dropped rather than assumed in-scope. See the printed count below.
year_filter = last_report_data['Year project started'].between(FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX).fillna(False)
n_missing_start_year = last_report_data['Year project started'].isna().sum()
print(f"NOTE: {n_missing_start_year} last_report_data rows have a missing/blank 'Year project started' "
      f"and are being EXCLUDED by the {FUNDING_START_YEAR_MIN}-{FUNDING_START_YEAR_MAX} start-year filter.")

last_report_data = last_report_data[year_filter].reset_index(drop=True)
last_report_data_edited = last_report_data.copy()

print(f"{n_loaded} rows loaded from '{LAST_REPORT_FILE}'")
print(f"{n_loaded - len(last_report_data)} rows dropped by the {FUNDING_START_YEAR_MIN}-{FUNDING_START_YEAR_MAX} start-year filter")
print(f"{len(last_report_data)} rows remain in last_report_data")
last_report_data.head()

NOTE: 25 last_report_data rows have a missing/blank 'Year project started' and are being EXCLUDED by the 2020-2025 start-year filter.
1678 rows loaded from 'Funding2026_inscope.xlsx'
251 rows dropped by the 2020-2025 start-year filter
1427 rows remain in last_report_data


,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,lrd_row_id
0,Denmark announces 1 billion kroner for plant-b...,NaN,NaN,airtable,1250000000,1250000000,DKK,177000000.0,177000000.0,162500000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,Plant2Food,The new collaborative platform Plant2Food will...,NaN,airtable,200000000,200000000,DKK,28473000.0,0.0,26000000.0,...,4333333.333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,CO2 as a sustainable raw material in our futur...,"In a new consortium, companies and university ...",NaN,airtable,100000000,100000000,DKK,27000000.0,0.0,13000000.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,"Precision Technology: Biotechnology, smart sen...",The project has three sub-goals:\n\nDeveloping...,NaN,airtable,64200000,64200000,NOK,NaN,NaN,5585400.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,NaN,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.0,...,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4


### 2. GFI Grants Tracker data (Airtable export)

In [3]:
grants_tracker_data = pd.read_excel(RAW_DATA_DIR / GRANTS_TRACKER_FILE)
grants_tracker_data = assign_stable_row_id(grants_tracker_data, 'gt_row_id')  # stamped before filtering, reflects the raw export's row position

grants_tracker_data['EXT_Date added'] = pd.to_datetime(grants_tracker_data['EXT_Date added'], errors='coerce')
grants_tracker_data['EXT_Year project starts'] = pd.to_numeric(grants_tracker_data['EXT_Year project starts'], errors='coerce')

date_filter = (
    (grants_tracker_data['EXT_Date added'] >= GT_DATE_ADDED_START) &
    (grants_tracker_data['EXT_Date added'] <= GT_DATE_ADDED_END)
)

# Only grants with a funding start year in [FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX] are in scope.
# NOTE: rows with a missing/blank 'EXT_Year project starts' are EXCLUDED here - a blank year can't
# be confirmed to fall in range, so it's dropped rather than assumed in-scope. See the printed count below.
year_filter = grants_tracker_data['EXT_Year project starts'].between(FUNDING_START_YEAR_MIN, FUNDING_START_YEAR_MAX).fillna(False)
n_missing_start_year = grants_tracker_data['EXT_Year project starts'].isna().sum()
print(f"NOTE: {n_missing_start_year} grants_tracker rows have a missing/blank 'EXT_Year project starts' "
      f"and are being EXCLUDED by the {FUNDING_START_YEAR_MIN}-{FUNDING_START_YEAR_MAX} start-year filter.")

if LAST_REPORT_SCOPE == 'europe':
    europe_filter = (
        (grants_tracker_data['EXT_Funder region'] == 'Europe') |
        (grants_tracker_data['EXT_PI organization region'] == 'Europe')
    )
    # Europe: incremental slice only (already covered historically). Non-Europe: full backlog,
    # any date, one-time catch-up. Blank/missing region ends up on the non-Europe side (~europe_filter
    # is True whenever both region columns are NaN, since `NaN == 'Europe'` evaluates False) -
    # included in the backfill per your call above.
    combined_filter = (europe_filter & date_filter) | ~europe_filter
elif LAST_REPORT_SCOPE == 'global':
    # last_report_data already has full worldwide coverage - just the incremental window applies.
    combined_filter = date_filter
else:
    raise ValueError(f"LAST_REPORT_SCOPE must be 'europe' or 'global', got {LAST_REPORT_SCOPE!r}")

combined_filter = combined_filter & year_filter  # start-year requirement applies regardless of LAST_REPORT_SCOPE

grants_tracker_data = grants_tracker_data[combined_filter].reset_index(drop=True)
print(f"{len(grants_tracker_data)} grants included after filtering")

# Rename Gov't -> Gov throughout the pipeline to avoid apostrophe quoting issues
grants_tracker_data = grants_tracker_data.rename(columns={
    "INT_Gov't contribution (actual currency)": 'INT_Gov contribution (actual currency)',
    "EXT_Gov't contribution (USD)":             'EXT_Gov contribution (USD)',
})

grants_tracker_data_edited = grants_tracker_data.copy()
grants_tracker_data.head()

NOTE: 118 grants_tracker rows have a missing/blank 'EXT_Year project starts' and are being EXCLUDED by the 2020-2025 start-year filter.
477 grants included after filtering


,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags,gt_row_id
0,Disruptive technologies and US food security: ...,262500,262500,USD,262500,262500.0,Awarded,https://cris.nifa.usda.gov/cgi-bin/starfinder/...,NaN,Erin wrote LoS.,...,2025-05-01T00:00:00.000Z,NaN,NaN,NaN,Achieving food security for a rapidly growing ...,grant.9216636,NaN,NaN,NaN,11
1,A Robust Biofilm-Biomat Reactor for Conversion...,749939,749939,USD,749939,749939.0,Awarded,https://www.sbir.gov/sbirsearch/detail/1670217,Sustainable Bioproducts is now Nature's Fynd.,Sustainable Bioproducts received other SBIR gr...,...,2022-01-06T00:00:00.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,14
2,Enabling Technology - Reducing Greenhouse Gas ...,249999,249999,USD,249999,249999.0,Awarded,https://www.sbir.gov/sbirsearch/detail/1703269,Cambridge Crops is now Mori.,NaN,...,2021-05-01T00:00:00.000Z,NaN,NaN,NaN,"Food production, and in particular animal-deri...",grant.9287038,NaN,NaN,NaN,15
3,Breeding Crops to Support the Plant-based Prot...,NaN,NaN,USD,NaN,NaN,Awarded,https://portal.nifa.usda.gov/web/crisprojectpa...,NaN,NaN,...,2025-10-01T00:00:00.000Z,NaN,NaN,NaN,Demand for animal protein is projected to rise...,grant.9216553,NaN,NaN,NaN,16
4,Functionalization of pea proteins using cold e...,476270,476270,USD,476270,476270.0,Awarded,https://cris.nifa.usda.gov/cgi-bin/starfinder/...,NaN,NaN,...,2024-01-01T00:00:00.000Z,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17


In [4]:
valid_countries = set()
for c in pycountry.countries:
    valid_countries.add(c.name.strip().lower())
    if hasattr(c, 'common_name'):
        valid_countries.add(c.common_name.strip().lower())

valid_countries.update({'czech republic', 'russia', 'turkey', 'uk'})

country_aliases = {'usa': 'United States', 'u.s.a.': 'United States', 'u.s.': 'United States'}

def _normalize_country_cell(val):
    if pd.isna(val) or str(val).strip() == '':
        return val
    parts = re.split(r'([;,])', str(val))
    parts[0] = country_aliases.get(parts[0].strip().lower(), parts[0])
    return ''.join(parts)

def _is_valid_country_cell(val):
    if pd.isna(val) or str(val).strip() == '':
        return False
    first_part = re.split(r'[;,]', str(val))[0].strip().lower()
    return first_part in valid_countries

for _df in (grants_tracker_data, grants_tracker_data_edited):
    _df['EXT_PI organization country'] = _df['EXT_PI organization country'].apply(_normalize_country_cell)

before_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
removed_vals = (
    grants_tracker_data['EXT_PI organization country']
    .dropna()
    .loc[lambda s: ~s.apply(_is_valid_country_cell)]
    .unique()
)

for _df in (grants_tracker_data, grants_tracker_data_edited):
    _df['EXT_PI organization country'] = _df['EXT_PI organization country'].apply(
        lambda val: val if _is_valid_country_cell(val) else None
    )

after_count = grants_tracker_data['EXT_PI organization country'].notna().sum()
print(f"Nulled out {before_count - after_count} non-country values ({before_count} -> {after_count} filled)")
print(f"Values removed: {sorted(removed_vals)}")

Nulled out 6 non-country values (468 -> 462 filled)
Values removed: ['10 countries', 'Maryland', 'Massachusetts', 'TBD', 'Washington']


### 3. Dimensions data (from S1's `{RUN_TABLE}`)

Reads S1's automated Dimensions grants query output directly from `funding.db`, replacing the 4 manually-exported Dimensions Excel files the original prototype used.

**Adapter cell below**: normalizes S1's DSL-native column names/shapes into the flat column names the rest of this notebook expects (matching the legacy manual-export format it was originally built around). If S1's field shapes change, only this cell should need updating - flagged in the implementation plan as needing live validation against a real `dsl.query()` call.

Also handles the case where `RUN_TABLE` already holds data in the legacy manual-export schema (e.g. a test table built directly from the raw `Dimensions-Grant-*.xlsx` files, which already has columns like `Grant ID`/`Researchers` rather than S1's lowercase `id`/`researchers`) - detected automatically, no adapting applied in that case since it's already in the target shape.

In [5]:
import ast

con = duckdb.connect(str(DB_PATH))
dimensions_raw = con.sql(f"SELECT * FROM {RUN_TABLE}").df()
con.close()
print(f"{len(dimensions_raw)} rows loaded from '{RUN_TABLE}'")

def _maybe_parse_literal(val):
    """Some Dimensions DSL fields with deeply nested structure (e.g. researchers, whose
    dicts contain a further nested research_orgs list) come back from DuckDB as a Python-repr
    string rather than a real list/dict - detect and parse that case back into real objects."""
    if isinstance(val, str) and val.strip().startswith(('[', '{')):
        try:
            return ast.literal_eval(val)
        except (ValueError, SyntaxError):
            return val
    return val


def _join_list(val):
    """Flatten a dimcli-style nested field (list of dicts, list of strings, or a plain
    scalar) into a semicolon-joined string, matching the legacy manual-export format."""
    val = _maybe_parse_literal(val)
    if val is None or (not isinstance(val, (list, tuple, np.ndarray)) and pd.isna(val)):
        return None
    if isinstance(val, str):
        return val
    if not isinstance(val, (list, tuple, np.ndarray)):
        return str(val)
    parts = []
    for item in val:
        if isinstance(item, dict):
            name = item.get('name')
            if name is None and ('first_name' in item or 'last_name' in item):
                name = f"{item.get('first_name', '')} {item.get('last_name', '')}".strip()
            parts.append(str(name if name is not None else item))
        else:
            parts.append(str(item))
    parts = [p for p in parts if p]
    return '; '.join(parts) if parts else None


def _first_or_none(val):
    joined = _join_list(val)
    if not joined:
        return None
    return joined.split(';')[0].strip()


def _derive_researchers_from_investigators(val):
    """Build a PI-first, semicolon-joined name list from Dimensions' investigators field
    (which has an explicit 'role' per person: 'PI'/'Co-PI'/etc.), rather than researchers
    (a flat, role-less contributor list that doesn't reliably put the actual PI first - or
    even include them at all in some cases)."""
    val = _maybe_parse_literal(val)
    if val is None or (not isinstance(val, (list, tuple, np.ndarray)) and pd.isna(val)):
        return None
    if not isinstance(val, (list, tuple, np.ndarray)):
        return None
    pis, others = [], []
    for item in val:
        if not isinstance(item, dict):
            continue
        name = f"{item.get('first_name') or ''} {item.get('last_name') or ''}".strip()
        if not name:
            continue
        (pis if item.get('role') == 'PI' else others).append(name)
    ordered = pis + others
    return '; '.join(ordered) if ordered else None


# Dimensions exposes 8 currency-converted funding amounts (no generic native-currency amount -
# that field was deprecated/removed from the grants DSL source in 2018). Where funding_currency
# matches one of these 8, we can back out a genuine native-currency figure; otherwise there's no
# way to get one via the API at all.
_CURRENCY_TO_FIELD = {
    'USD': 'funding_usd', 'EUR': 'funding_eur', 'GBP': 'funding_gbp', 'AUD': 'funding_aud',
    'CAD': 'funding_cad', 'CHF': 'funding_chf', 'JPY': 'funding_jpy', 'NZD': 'funding_nzd',
}

def _native_currency_amount(row):
    currency = row.get('funding_currency')
    if currency is None or pd.isna(currency):
        return None
    field = _CURRENCY_TO_FIELD.get(str(currency).strip().upper())
    return row.get(field) if field else None


if 'id' in dimensions_raw.columns:
    # S1's DSL-schema output - needs adapting into the legacy flat column shape
    print("Detected S1 (DSL) schema - applying adapter.")
    dimensions_data = pd.DataFrame({
        'Grant ID': dimensions_raw['id'],
        'Title translated': dimensions_raw.get('title'),
        'Title': dimensions_raw.get('original_title'),
        'Abstract translated': dimensions_raw.get('abstract'),
        'Abstract': None,  # not queried by S1 - grants DSL has no separate original-language abstract field
        'Researchers': dimensions_raw['investigators'].apply(_derive_researchers_from_investigators) if 'investigators' in dimensions_raw else None,
        'Research Organization - standardized': dimensions_raw['research_org_names'].apply(_join_list) if 'research_org_names' in dimensions_raw else None,
        # Currency/amount columns: 'Total amount' is the grant's amount in its own native
        # currency, backed out via _native_currency_amount above where funding_currency matches
        # one of the 8 Dimensions exposes - None otherwise (no API field can give us this for e.g.
        # NOK/SEK-funded grants). 'Total amount (USD)'/'(EUR)' are Dimensions' direct conversions.
        'Currency': dimensions_raw.get('funding_currency'),
        'Total amount': dimensions_raw.apply(_native_currency_amount, axis=1) if len(dimensions_raw) else None,
        'Total amount (USD)': dimensions_raw.get('funding_usd'),
        'Total amount (EUR)': dimensions_raw.get('funding_eur'),
        'Start date': dimensions_raw.get('start_date'),
        'Start Year': dimensions_raw.get('start_year'),
        'End Year': pd.to_datetime(dimensions_raw['end_date'], errors='coerce').dt.year if 'end_date' in dimensions_raw else None,
        'State of standardized research organization': None,  # not queried by S1
        'Country of standardized research organization': dimensions_raw['research_org_countries'].apply(_join_list) if 'research_org_countries' in dimensions_raw else None,
        'Funder': dimensions_raw['funder_org_name'].apply(_join_list) if 'funder_org_name' in dimensions_raw else None,
        'Funder Country': dimensions_raw['funder_org_countries'].apply(_join_list) if 'funder_org_countries' in dimensions_raw else None,
        'Source Linkout': dimensions_raw['linkout'].apply(_first_or_none) if 'linkout' in dimensions_raw else None,
    })
elif 'Grant ID' in dimensions_raw.columns:
    # Already in the legacy manual-export schema (e.g. a test table loaded straight from the
    # raw Dimensions-Grant-*.xlsx files) - nothing to adapt.
    print("Detected legacy manual-export schema - using as-is, no adapting needed.")
    dimensions_data = dimensions_raw.copy()
else:
    raise ValueError(
        f"'{RUN_TABLE}' has neither an 'id' column (S1/DSL schema) nor a 'Grant ID' column "
        f"(legacy manual-export schema) - can't tell which format this table is in."
    )

# Defensive dedup - S1 already dedupes by id, but cheap to re-check across the S1/S2 boundary
before = len(dimensions_data)
dimensions_data = dimensions_data.drop_duplicates(subset='Grant ID').reset_index(drop=True)
print(f"Removed {before - len(dimensions_data)} duplicates ({before} -> {len(dimensions_data)} rows)")

dimensions_data.head()

5052 rows loaded from 'run_260727_0922'
Detected S1 (DSL) schema - applying adapter.
Removed 0 duplicates (5052 -> 5052 rows)


,Grant ID,Title translated,Title,Abstract translated,Abstract,Researchers,Research Organization - standardized,Currency,Total amount,Total amount (USD),Total amount (EUR),Start date,Start Year,End Year,State of standardized research organization,Country of standardized research organization,Funder,Funder Country,Source Linkout
0,grant.15119262,Characterization of the protein concentrate ob...,Caracterização do concentrado proteico obtido ...,With the growing demand for fresh and natural ...,None,Isabelle Neves; Júlia Gomes,Federal University of São Carlos,NaN,NaN,NaN,NaN,2025-12-27,2025,2026.0,None,Brazil; Brazil,São Paulo Research Foundation,Brazil,http://bv.fapesp.br/pt/bolsas/232113/caracteri...
1,grant.15151354,3D Human Trigeminal Ganglion Organoids: The Et...,3D Human Trigeminal Ganglion Organoids: The Et...,Pain research is one of the most ethically com...,None,Pierfrancesco Pagella,Linköping University,SEK,NaN,243508.0,208589.0,2025-12-01,2025,2028.0,None,Sweden,Swedish Research Council,Sweden,https://www.vr.se/swecris.html#/project/2025-0...
2,grant.15150934,From Potato Waste to Mycoprotein – A Sustainab...,From Potato Waste to Mycoprotein – A Sustainab...,This project aims to upcycle food-grade potato...,None,Carl Grey,Lund University,SEK,NaN,432546.0,370690.0,2025-12-01,2025,2027.0,None,Sweden,Swedish Research Council for Environment Agric...,Sweden,https://www.vr.se/swecris.html#/project/2025-0...
3,grant.15149823,Advancing 3D plant-based scaffolds for culture...,Advancing 3D plant-based scaffolds for culture...,The demand for protein-rich foods is rapidly g...,None,NaN,University of Idaho,USD,606620.0,606620.0,519511.0,2025-12-01,2025,2029.0,None,United States,National Institute of Food and Agriculture,United States,https://portal.nifa.usda.gov/web/crisprojectpa...
4,grant.15149598,PARTNERSHIP: Development of lipid- and protein...,PARTNERSHIP: Development of lipid- and protein...,Global food systems face mounting pressure as ...,None,NaN,"University of California, Davis",USD,751937.0,751937.0,644133.0,2025-12-01,2025,2028.0,None,United States,National Institute of Food and Agriculture,United States,https://portal.nifa.usda.gov/web/crisprojectpa...


#### Dimensions -> last report's data, exact ID match

Gap-fills the last report's data from Dimensions (never overwrites non-empty cells, except funding columns where an existing 0 is treated as fillable), then removes matched rows from `dimensions_data` - no manual review needed, this is a real business-key match.

In [6]:
col_map = {
    'Grant ID':                                       'Identification code',  # only fills if currently empty - never overwrites the ID join key used above
    'Title translated':                              'Title',
    'Title':                                         'Original title',
    'Abstract translated':                           'Abstract',
    'Total amount':                                  'Total amount',
    'Currency':                                      'Currency',
    'Total amount (USD)':                            'Total amount (USD)',
    'Total amount (EUR)':                            'Total amount (EUR)',
    'Start date':                                    'Project start date',
    'Start Year':                                    'Year project started',
    'End Year':                                      'End date',
    'State of standardized research organization':   'PI organisation state',
    'Country of standardized research organization': 'PI organisation country',
    'Funder':                                        'Funder name',
    'Funder Country':                                'Funder Country',
    'Source Linkout':                                'URL for announcement',
}
funding_cols_lrd = {'Total amount', 'Total amount (USD)', 'Total amount (EUR)'}

lrd_id_map = {}
for idx, val in last_report_data['Identification code'].items():
    if not is_empty(val):
        lrd_id_map.setdefault(str(val).strip(), []).append(idx)

matched_ids = set()
changed_indices = set()
cells_filled = 0

for _, dim_row in dimensions_data.iterrows():
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in lrd_id_map:
        continue
    matched_ids.add(grant_id)
    for lrd_idx in lrd_id_map[grant_id]:
        cells_filled += gap_fill(dim_row, last_report_data_edited, lrd_idx, col_map, funding_cols_lrd, changed_indices)
        cells_filled += gap_fill_researchers_and_orgs(
            dim_row, last_report_data_edited, lrd_idx,
            pi_col='Project lead (PI)', collab_col='Collaborator names',
            org_pi_col='PI organisation', org_collab_col='Collaborator institutions',
            changed_indices=changed_indices,
        )

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Matched {len(matched_ids)} grants with last_report_data")
print(f"Filled {cells_filled} missing values across {len(changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Matched 550 grants with last_report_data
Filled 845 missing values across 510 rows
Removed 550 rows from dimensions_data (4502 remaining)


In [7]:
export_highlighted_diff(last_report_data, last_report_data_edited, changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_dim_ID_matches_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,lrd_row_id
5,National Alternative Proteins Innovation and K...,"To secure a continued supply of safe, tasty, ...",NAPIC: National Alternative Protein Innovation...,airtable,38000000,16001352,GBP,48859450.0,18697500.0,45220000.00,...,3.173601e+06,3.173601e+06,3173601.480,NaN,NaN,NaN,NaN,NaN,NaN,5
8,Green technology for plant-based food (GreenPl...,To maintain national food self-sufficiency and...,GreenPlantFood - Green technology for plant-ba...,airtable,27600000,27600000,NOK,2604048.0,2604048.0,2401200.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8
10,University of Bath Cellular Agriculture Manufa...,Imagine being able to manufacture food anywher...,Cellular Agriculture Manufacturing Hub,airtable,21214469,12302109,GBP,26130285.0,15655848.0,25245218.11,...,1.829939e+06,1.829939e+06,1829938.714,1829938.714,NaN,NaN,NaN,NaN,NaN,10
11,ARRIVAL (ARRIVAL of Cellular Agriculture-Enabl...,Global food production must increase more than...,KSP: ARRIVAL of cellular agriculture- Enabling...,airtable,20000000,20000000,NOK,1993440.0,1993440.0,1740000.00,...,3.480000e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11
15,Microbial Food Hub,We propose the creation of an Engineering Bio...,Engineering Biology Hub for Microbial Food,airtable,14000000,12634909,GBP,17427480.0,14937840.0,16660000.00,...,2.505924e+06,2.505924e+06,2505923.618,NaN,NaN,NaN,NaN,NaN,NaN,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1286,Enhancing biodiversity and ecosystem services ...,Background and rationale: Agricultural product...,Enhancing biodiversity and ecosystem services ...,2025 Dimensions gen,749801,749801,CHF,841774.0,NaN,802287.07,...,2.005718e+05,2.005718e+05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1533
1287,Food workers of the world unite! A qualitative...,Our contemporary food system is failing us: it...,Food workers of the world unite! A qualitative...,2025 Dimensions gen,49554,49554,EUR,52591.0,NaN,49554.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1534
1288,Badanie potencjału kompleksów białkowo-pektyno...,The aging population is a global problem of th...,Investigating the potential of protein-pectin ...,2025 Dimensions gen,210000,NaN,PLN,52868.0,NaN,48300.00,...,0.000000e+00,0.000000e+00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1535
1289,Designing a manufacturing facility of the futu...,The UN's goal of eliminating hunger will be mi...,Designing a manufacturing facility of the futu...,2025 Dimensions gen,74716,74716,GBP,94487.0,NaN,88912.04,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1536


#### Dimensions -> grants tracker, exact ID match

In [8]:
gt_col_map = {
    'Title translated':                              'EXT_Title',
    'Abstract translated':                           'EXT_Abstract',
    'Total amount':                                  'INT_Total amount (actual currency)',
    'Currency':                                      'INT_Currency type',
    'Total amount (USD)':                            'EXT_Total amount (USD)',
    'Start date':                                    'EXT_Project start date (estimated)',
    'End Year':                                      'INT_End date',
    'State of standardized research organization':   'EXT_PI organisation state',
    'Country of standardized research organization': 'EXT_PI organization country',
    'Funder':                                        'EXT_Funder name',
    'Funder Country':                                'EXT_Funder country',
    'Source Linkout':                                'EXT_URL for announcement',
}
funding_cols_gt = {'INT_Total amount (actual currency)', 'EXT_Total amount (USD)'}

gt_id_map = {}
for idx, val in grants_tracker_data['Dimensions.ai grant ID'].items():
    if not is_empty(val):
        gt_id_map.setdefault(str(val).strip(), []).append(idx)

gt_matched_ids = set()
gt_changed_indices = set()
gt_cells_filled = 0

for _, dim_row in dimensions_data.iterrows():
    grant_id = str(dim_row['Grant ID']).strip()
    if grant_id not in gt_id_map:
        continue
    gt_matched_ids.add(grant_id)
    for gt_idx in gt_id_map[grant_id]:
        gt_cells_filled += gap_fill(dim_row, grants_tracker_data_edited, gt_idx, gt_col_map, funding_cols_gt, gt_changed_indices)
        gt_cells_filled += gap_fill_researchers_and_orgs(
            dim_row, grants_tracker_data_edited, gt_idx,
            pi_col='EXT_Project lead (PI)', collab_col='EXT_Collaborator names',
            org_pi_col='EXT_PI organization', org_collab_col='EXT_Collaborator organizations',
            changed_indices=gt_changed_indices,
        )

before = len(dimensions_data)
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).str.strip().isin(gt_matched_ids)
].reset_index(drop=True)
after = len(dimensions_data)

print(f"Matched {len(gt_matched_ids)} grants with grants_tracker_data")
print(f"Filled {gt_cells_filled} missing values across {len(gt_changed_indices)} rows")
print(f"Removed {before - after} rows from dimensions_data ({after} remaining)")

Matched 150 grants with grants_tracker_data
Filled 40 missing values across 35 rows
Removed 150 rows from dimensions_data (4352 remaining)


In [9]:
export_highlighted_diff(grants_tracker_data, grants_tracker_data_edited, gt_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_dim_ID_matches_gt_data.xlsx')

,EXT_Title,INT_Total amount (actual currency),INT_Gov contribution (actual currency),INT_Currency type,EXT_Total amount (USD),EXT_Gov contribution (USD),EXT_Funding decision,EXT_URL for announcement,EXT_Notes (external),INT_Notes INTERNAL ONLY,...,INT_End Date (Formula),INT_Funding call,INT_Success rate,INT_Minority serving institution?,EXT_Abstract,Dimensions.ai grant ID,Program type,Research area,Flags,gt_row_id
205,Reduction of off-flavors and improvement of co...,45000,45000,CAD,33504,33504.0,Awarded,https://drive.google.com/file/d/1X-ZldjgmPsHG1...,"Original award amount is CAD 45,000. Co-funded...",NaN,...,2024-01-01T00:00:00.000Z,NaN,NaN,NaN,Plant protein is a potential environmentally f...,grant.14458090,NaN,NaN,NaN,387
227,New Arrangement for Research and Innovation in...,5700000,5700000,BRL,1147297,1147297.0,Awarded,https://gfi.org.br/parana-vai-investir-r5-7-mi...,"Original award amount is BRL 5,700,000.",NaN,...,2027-04-20T00:00:00.000Z,NaN,NaN,NaN,Research project to evaluate innovations in al...,grant.14003374,NaN,NaN,NaN,446
267,Analogs of chicken cuts from tissue engineerin...,0,NaN,BRL,0,NaN,Awarded,https://sucupira.capes.gov.br/sucupira/public/...,NaN,NaN,...,2022-10-01T00:00:00.000Z,NaN,NaN,NaN,The food industry has presented a wide range o...,grant.14003555,VETERINARY SCIENCES (40014010042P9) (Program),Cell lines,NaN,710
268,Laying the Scientific and Engineering Foundati...,2400000,NaN,USD,2400000,NaN,Awarded,http://www.nsf.gov/awardsearch/showAward?AWD_I...,NaN,NaN,...,2026-09-15T00:00:00.000Z,NaN,NaN,NaN,Cultivated meat could be a cornerstone of a su...,grant.13610700,Continuing Grant (Award Instrument); Cellular ...,Cell lines,NaN,712
271,I-Corps: Unlocking constraints on mammalian ce...,50000,NaN,USD,50000,NaN,Awarded,http://www.nsf.gov/awardsearch/showAward?AWD_I...,NaN,NaN,...,2025-02-15T00:00:00.000Z,NaN,NaN,NaN,The broader impact/commercial potential of thi...,grant.13242381,Standard Grant (Award Instrument); I-Corps (Pr...,Cell lines,NaN,719
272,GCR: Laying the Scientific and Engineering Fou...,3549236,NaN,USD,3549236,NaN,Awarded,http://www.nsf.gov/awardsearch/showAward?AWD_I...,NaN,NaN,...,2025-10-01T00:00:00.000Z,NaN,NaN,NaN,This proposal is focused on convergent researc...,grant.9398256,Continuing Grant (Award Instrument); GCR-Growi...,Cell lines,NaN,720
273,EAGER: 3D Printing of Aligned Muscle Fibers fo...,308000,NaN,USD,308000,NaN,Awarded,http://www.nsf.gov/awardsearch/showAward?AWD_I...,NaN,NaN,...,2024-09-15T00:00:00.000Z,NaN,NaN,NaN,The continued growth of human populations with...,grant.12942760,Standard Grant (Award Instrument); AM-Advanced...,Cell lines,NaN,726
274,ECJ - Characterization of the growth and diffe...,0,NaN,GBP,0,NaN,Awarded,https://sucupira.capes.gov.br/sucupira/public/...,NaN,NaN,...,2023-03-01T00:00:00.000Z,NaN,NaN,NaN,The consumption of animal protein by all manki...,grant.13398924,CELL BIOLOGY (32001010009P8) (Program),Cell lines,NaN,729
278,Use of induced pluripotent cells (iPSCs) as an...,0,NaN,BRL,0,NaN,Awarded,http://bv.fapesp.br/pt/bolsas/206607/utilizaca...,NaN,NaN,...,2024-04-03T00:00:00.000Z,NaN,NaN,NaN,Alternative and innovative technologies applie...,grant.13658438,Scholarships Abroad - Research Internship - Do...,Cell lines,NaN,738
279,Research on food sanitation hazards and risks ...,15408000,NaN,JPY,115805,NaN,Awarded,https://mhlw-grants.niph.go.jp/project/165295,NaN,NaN,...,2024-04-01T00:00:00.000Z,NaN,NaN,NaN,Food tech is a technology that utilizes cuttin...,grant.13306235,tbd,Cell lines,NaN,740


### 4. Grants Tracker vs last report's data
#### a. Dimensions ID match

In [10]:
last_report_data_pre_gt = last_report_data_edited.copy()

gt_lrd_col_map = {
    'EXT_Title':                              'Title',
    'EXT_Abstract':                           'Abstract',
    'INT_Total amount (actual currency)':     'Total amount',
    'INT_Gov contribution (actual currency)': 'Gov contribution',
    'INT_Currency type':                      'Currency',
    'EXT_Total amount (USD)':                 'Total amount (USD)',
    'EXT_Gov contribution (USD)':             'Gov contribution (USD)',
    'EXT_Funding decision':                   'Funding decision',
    'EXT_Project start date (estimated)':     'Project start date',
    'INT_End date':                           'End date',
    'EXT_Project lead (PI)':                  'Project lead (PI)',
    'EXT_PI organization':                    'PI organisation',
    'EXT_PI organization type':               'PI organisation type',
    'EXT_PI organization country':            'PI organisation country',
    'EXT_PI organization region':              'PI organisation region',
    'PI organisation state':                  'PI organisation state',
    'EXT_Collaborator names':                 'Collaborator names',
    'EXT_Collaborator organizations':         'Collaborator institutions',
    'EXT_Funder name':                        'Funder name',
    'EXT_Funder type':                        'Funder type',
    'EXT_Funder country':                     'Funder Country',
    'EXT_Funder region':                      'Funder region',
    'EXT_Production platform':                'AP pillar',
    'EXT_End product type':                   'End product type',
    'EXT_Award purpose':                      'Award purpose',
    'EXT_URL for announcement':               'URL for announcement',
    'EXT_Date added':                         'Date added',
    'EXT_Last modified':                      'Last modified',
    'EXT_Years project starts':               'Year project started',
    'INT_GFI grantee?':                       'GFI grantee',
    'INT_GFI Los?':                            'GFI LOS',
    'INT_Link to Los':                        'Link to LOS',
    'INT_GFI partner?':                       'GFI partner',
    'INT_Tier':                                'Tier',
}
funding_cols_lrd2 = {'Total amount', 'Total amount (USD)', 'Gov contribution', 'Gov contribution (USD)'}

# Only Dimensions' own 'grant.NNNNNNN' ID format is trustworthy for matching - anything else
# (blanks, free text like 'need to check', manually-entered non-Dimensions codes) is skipped
# rather than treated as a shared ID, since it isn't a real Dimensions grant ID on either side.
_GRANT_ID_RE = re.compile(r'^grant\.\d+$')

lrd2_id_map = {}
for idx, val in last_report_data_edited['Identification code'].items():
    if not is_empty(val) and _GRANT_ID_RE.match(str(val).strip()):
        lrd2_id_map.setdefault(str(val).strip(), []).append(idx)

gt_lrd_matched_ids = set()
gt_lrd_changed_indices = set()
gt_lrd_cells_filled = 0

for _, gt_row in grants_tracker_data_edited.iterrows():
    grant_id = str(gt_row['Dimensions.ai grant ID']).strip()
    if not _GRANT_ID_RE.match(grant_id) or grant_id not in lrd2_id_map:
        continue
    gt_lrd_matched_ids.add(grant_id)
    for lrd_idx in lrd2_id_map[grant_id]:
        gt_lrd_cells_filled += gap_fill(gt_row, last_report_data_edited, lrd_idx, gt_lrd_col_map, funding_cols_lrd2, gt_lrd_changed_indices)

before = len(grants_tracker_data_edited)
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['Dimensions.ai grant ID'].astype(str).str.strip().isin(gt_lrd_matched_ids)
].reset_index(drop=True)
after = len(grants_tracker_data_edited)

print(f"Matched {len(gt_lrd_matched_ids)} grants between grants_tracker_data_edited and last_report_data_edited")
print(f"Filled {gt_lrd_cells_filled} missing values across {len(gt_lrd_changed_indices)} rows")
print(f"Removed {before - after} rows from grants_tracker_data_edited ({after} remaining)")

Matched 12 grants between grants_tracker_data_edited and last_report_data_edited
Filled 27 missing values across 8 rows
Removed 12 rows from grants_tracker_data_edited (465 remaining)


In [11]:
# OPTIONAL - shows which grants were duplicated within the grants tracker data
dupes = grants_tracker_data[
    grants_tracker_data['Dimensions.ai grant ID'].isin(gt_lrd_matched_ids)
    & grants_tracker_data['Dimensions.ai grant ID'].duplicated(keep=False)
][['Dimensions.ai grant ID', 'EXT_Title']].sort_values('Dimensions.ai grant ID')
dupes

,Dimensions.ai grant ID,EXT_Title


In [12]:
export_highlighted_diff(last_report_data_pre_gt, last_report_data_edited, gt_lrd_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_gt_changes_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,lrd_row_id
807,Exploring Mycoprotein Potential: Assessing Bio...,The growing challenges of climate change and g...,Exploring Mycoprotein Potential: Assessing Bio...,Dimensions,125000,100000,GBP,0.0,NaN,148750.00,...,23800.0,23800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,825
889,Development of edible hollow fibre bioreactor ...,According to the Intergovernmental Panel on Cl...,Development of edible hollow fibre bioreactor ...,Dimensions,243119,243119,GBP,310592.0,NaN,289311.61,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,908
891,PRIME - Protein Recombination Improvement thro...,Addressing the urgent need to sustainably feed...,PRIME - Protein Recombination Improvement thro...,Dimensions,242532,242532,GBP,309845.0,NaN,288613.08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,910
894,The University of Edinburgh and Roslin Technol...,To create cultivated meat product prototypes (...,The University of Edinburgh and Roslin Technol...,Dimensions,149871,149871,GBP,191459.0,NaN,178346.49,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,913
1256,Epigenetic Reprogramming for Cellular Agricult...,Cellular agriculture has the potential to revo...,Epigenetic Reprogramming for Cellular Agricult...,2025 Dimensions CM,904867,904867,EUR,960340.0,NaN,904867.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1503
1257,3D Bioprinting of cultivated meat: Biopolymer ...,Due to disruptive effects of conventional mass...,3D Bioprinting of cultivated meat: Biopolymer ...,2025 Dimensions CM,320000,320000,EUR,339617.0,NaN,320000.00,...,64000.0,64000.0,64000.0,NaN,NaN,NaN,NaN,NaN,NaN,1504
1258,Engineered bionanoparticles as carriers for gr...,Cultured or artificial meat is a form of cellu...,Engineered bionanoparticles as carriers for gr...,2025 Dimensions CM,0,NaN,NaN,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1505
1265,Sustainable production of microbial proteins a...,As demand for animal origin food products incr...,Sustainable production of microbial proteins a...,2025 Dimensions F,300000,300000,EUR,318365.0,NaN,300000.00,...,60000.0,60000.0,60000.0,NaN,NaN,NaN,NaN,NaN,NaN,1512


#### b. Title-based match - REVIEW POINT 1 (exact title)

In [13]:
lrd_title_map = {}
for idx, val in last_report_data_edited['Title'].items():
    norm = normalize_title(val)
    if norm:
        lrd_title_map.setdefault(norm, []).append(idx)

title_match_rows = []
for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    norm = normalize_title(gt_row.get('EXT_Title'))
    if norm and norm in lrd_title_map:
        for lrd_idx in lrd_title_map[norm]:
            title_match_rows.append({
                'gt_row_id':               gt_row['gt_row_id'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'GT index':                gt_idx,
                'LRD index':               lrd_idx,
                'Title':                   gt_row.get('EXT_Title'),
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'GT Funder':               gt_row.get('EXT_Funder name'),
                'LRD Funder':              last_report_data_edited.at[lrd_idx, 'Funder name'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'GT Project start date':   gt_row.get('EXT_Project start date (estimated)'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

title_matches_df = pd.DataFrame(title_match_rows, columns=[
    'gt_row_id', 'lrd_row_id', 'GT index', 'LRD index', 'Title', 'GT Dimensions ID',
    'LRD Identification code', 'GT Funder', 'LRD Funder', 'GT Total (USD)', 'LRD Total (USD)',
    'GT Project start date', 'LRD Project start date',
])  # explicit columns so a zero-match run still has the right shape (not 0 columns)
print(f"{len(title_matches_df)} (GT, LRD) pairs found")
title_matches_df

14 (GT, LRD) pairs found


,gt_row_id,lrd_row_id,GT index,LRD index,Title,GT Dimensions ID,LRD Identification code,GT Funder,LRD Funder,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date
0,1853,1446,400,1199,Understanding the mechanisms associated with q...,grant.8655803,grant.9440850,Natural Sciences and Engineering Research Coun...,UK Research and Innovation (UKRI),21100,136774.0,2025-04-19,2025-03-24
1,1946,1489,438,1242,ANALEGS.CAT,NaN,NaN,Operation 16.01.01 (Cooperation for Innovation...,Catalan Rural Development Programme,273972,273972.0,2021-07-01,2021-07-01
2,1947,1490,439,1243,DGINN VEGEXT,NaN,NaN,Generalitat Valenciana,Generalitat Valenciana,337950,337950.0,2021-09-09,2021-09-09
3,1948,1491,440,1244,Proteinleg,NaN,NaN,European Agricultural Fund for Rural Developme...,European Agricultural Fund for Rural Developme...,625910,625910.0,NaT,NaT
4,1950,1493,442,1246,Texipro,NaN,NaN,French Government,Ministry of Agriculture and Food Sovereignty,1318005,1318005.0,2024-06-28,2024-06-28
5,1951,1494,443,1247,Innov’Légumineuses,NaN,NaN,French Government,Ministry of Agriculture and Food Sovereignty,1318005,1318005.0,2024-06-28,2024-06-28
6,1952,1495,444,1248,CDTI Misiones - Biotech Foods SL,NaN,NaN,CDTI Misiones,CDTI Misiones,1024018,1024018.0,1905-07-16,1905-07-16
7,1953,1496,445,1249,CDTI Misiones - Agrenvec SL,NaN,NaN,CDTI Misiones,CDTI Misiones,345446,345446.0,1905-07-16,1905-07-16
8,1954,1497,446,1250,CDTI Misiones - Aves Nobles y Derivados SL,NaN,NaN,CDTI Misiones,CDTI Misiones,233560,233560.0,1905-07-16,1905-07-16
9,1955,1498,447,1251,CDTI Misiones - Fertinagro Biotech SL,NaN,NaN,CDTI Misiones,CDTI Misiones,185027,185027.0,1905-07-16,1905-07-16


In [14]:
export_for_review(
    title_matches_df, id_cols=['gt_row_id', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_title_match_for_review.csv',
)

Saved 14 candidate matches for review -> data_review\260727_0922_gt_title_match_for_review.csv


,match_key,gt_row_id,lrd_row_id,GT index,LRD index,Title,GT Dimensions ID,LRD Identification code,GT Funder,LRD Funder,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date,is_true_match
0,1853__1446,1853,1446,400,1199,Understanding the mechanisms associated with q...,grant.8655803,grant.9440850,Natural Sciences and Engineering Research Coun...,UK Research and Innovation (UKRI),21100,136774.0,2025-04-19,2025-03-24,True
1,1946__1489,1946,1489,438,1242,ANALEGS.CAT,NaN,NaN,Operation 16.01.01 (Cooperation for Innovation...,Catalan Rural Development Programme,273972,273972.0,2021-07-01,2021-07-01,True
2,1947__1490,1947,1490,439,1243,DGINN VEGEXT,NaN,NaN,Generalitat Valenciana,Generalitat Valenciana,337950,337950.0,2021-09-09,2021-09-09,True
3,1948__1491,1948,1491,440,1244,Proteinleg,NaN,NaN,European Agricultural Fund for Rural Developme...,European Agricultural Fund for Rural Developme...,625910,625910.0,NaT,NaT,True
4,1950__1493,1950,1493,442,1246,Texipro,NaN,NaN,French Government,Ministry of Agriculture and Food Sovereignty,1318005,1318005.0,2024-06-28,2024-06-28,True
5,1951__1494,1951,1494,443,1247,Innov’Légumineuses,NaN,NaN,French Government,Ministry of Agriculture and Food Sovereignty,1318005,1318005.0,2024-06-28,2024-06-28,True
6,1952__1495,1952,1495,444,1248,CDTI Misiones - Biotech Foods SL,NaN,NaN,CDTI Misiones,CDTI Misiones,1024018,1024018.0,1905-07-16,1905-07-16,True
7,1953__1496,1953,1496,445,1249,CDTI Misiones - Agrenvec SL,NaN,NaN,CDTI Misiones,CDTI Misiones,345446,345446.0,1905-07-16,1905-07-16,True
8,1954__1497,1954,1497,446,1250,CDTI Misiones - Aves Nobles y Derivados SL,NaN,NaN,CDTI Misiones,CDTI Misiones,233560,233560.0,1905-07-16,1905-07-16,True
9,1955__1498,1955,1498,447,1251,CDTI Misiones - Fertinagro Biotech SL,NaN,NaN,CDTI Misiones,CDTI Misiones,185027,185027.0,1905-07-16,1905-07-16,True


**Pause here.** Open the exported CSV in `data/`, review the pre-filled `is_true_match` column (defaults to `True` = confirmed match -> gap-fill + remove), flip any false positives to `False` (kept, falls through to the fuzzy-match step below), save it as `..._reviewed.csv` in the same folder (change from '..._for_review'). Then continue.

Guidance: any grants that have a different start data, funding amount, dimensions ID, or other unique distinguishing feature should be switched to FALSE to prevent deduplication and removal.

In [15]:
last_report_data_pre_title = last_report_data_edited.copy()

confirmed_title, rejected_title = apply_reviewed_decisions(
    title_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_title_match_reviewed.csv',
    id_cols=['gt_row_id', 'lrd_row_id'],
)
print(f"{len(confirmed_title)} confirmed, {len(rejected_title)} rejected")

title_changed_indices = set()
title_cells_filled = 0

for _, match in confirmed_title.iterrows():
    gt_idx = int(match['GT index'])
    lrd_idx = int(match['LRD index'])
    gt_row = grants_tracker_data_edited.loc[gt_idx]
    title_cells_filled += gap_fill(gt_row, last_report_data_edited, lrd_idx, gt_lrd_col_map, funding_cols_lrd2, title_changed_indices)

confirmed_gt_row_ids = set(confirmed_title['gt_row_id'])
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['gt_row_id'].isin(confirmed_gt_row_ids)
].reset_index(drop=True)

print(f"Filled {title_cells_filled} missing values across {len(title_changed_indices)} rows")
print(f"Removed {len(confirmed_gt_row_ids)} confirmed matches from grants_tracker_data_edited ({len(grants_tracker_data_edited)} remaining)")

13 confirmed, 1 rejected
Filled 0 missing values across 0 rows
Removed 13 confirmed matches from grants_tracker_data_edited (452 remaining)


In [16]:
export_highlighted_diff(last_report_data_pre_title, last_report_data_edited, title_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_gt_title-match_changes_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,lrd_row_id


#### c. Partial title-based match - REVIEW POINT 2 (fuzzy title)

Uses `rapidfuzz.token_sort_ratio`: tokens are sorted before comparing, so word-order differences score highly. Gap-filling is intentionally skipped for partial matches - not reliable enough to fill data across - so a confirmed match here just removes the row (it's a duplicate, already tracked); a rejected one is kept and later appended as new data.

In [17]:
try:
    from rapidfuzz import fuzz as _rfuzz
except ImportError:
    raise ImportError("rapidfuzz not found - run: conda install -c conda-forge rapidfuzz")

lrd_title_list = [
    (idx, normalize_title(val))
    for idx, val in last_report_data_edited['Title'].items()
    if not is_empty(val)
]

partial_match_rows = []
for gt_idx, gt_row in grants_tracker_data_edited.iterrows():
    gt_norm = normalize_title(gt_row.get('EXT_Title'))
    if not gt_norm:
        continue
    for lrd_idx, lrd_norm in lrd_title_list:
        score = _rfuzz.token_sort_ratio(gt_norm, lrd_norm)
        if score >= FUZZY_THRESHOLD:
            partial_match_rows.append({
                'gt_row_id':               gt_row['gt_row_id'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'GT index':                gt_idx,
                'LRD index':               lrd_idx,
                'Similarity':              score,
                'GT Title':                gt_row.get('EXT_Title'),
                'LRD Title':               last_report_data_edited.at[lrd_idx, 'Title'],
                'GT Dimensions ID':        gt_row.get('Dimensions.ai grant ID'),
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'GT Total (USD)':          gt_row.get('EXT_Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'GT Project start date':   gt_row.get('EXT_Project start date (estimated)'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

partial_matches_df = pd.DataFrame(partial_match_rows, columns=[
    'gt_row_id', 'lrd_row_id', 'GT index', 'LRD index', 'Similarity', 'GT Title', 'LRD Title',
    'GT Dimensions ID', 'LRD Identification code', 'GT Total (USD)', 'LRD Total (USD)',
    'GT Project start date', 'LRD Project start date',
]).sort_values('Similarity', ascending=False).reset_index(drop=True)  # explicit columns so a zero-match run still has the right shape
print(f"{len(partial_matches_df)} (GT, LRD) pairs found at threshold {FUZZY_THRESHOLD}")
partial_matches_df

6 (GT, LRD) pairs found at threshold 85


,gt_row_id,lrd_row_id,GT index,LRD index,Similarity,GT Title,LRD Title,GT Dimensions ID,LRD Identification code,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date
0,1239,1446,326,1199,100.000000,Understanding the mechanisms associated\n wit...,Understanding the mechanisms associated with q...,grant.14440689,grant.9440850,20734,136774.0,2023-04-01,2025-03-24
1,1621,1446,336,1199,100.000000,Understanding the\n mechanisms associated wit...,Understanding the mechanisms associated with q...,grant.13004193,grant.9440850,22329,136774.0,2021-04-01,2025-03-24
2,1655,1446,347,1199,100.000000,Understanding the\n mechanisms associated wit...,Understanding the mechanisms associated with q...,grant.9871075,grant.9440850,42390,136774.0,2020-04-01,2025-03-24
3,1853,1446,400,1199,100.000000,Understanding the mechanisms associated with q...,Understanding the mechanisms associated with q...,grant.8655803,grant.9440850,21100,136774.0,2025-04-19,2025-03-24
4,1949,1492,438,1245,100.000000,ProxiMed - Exploration and Implementation of P...,ProxiMed - Exploration and Implementation of P...,NaN,NaN,4613806,4613806.0,1905-07-14,1905-07-14
5,1912,1475,420,1228,88.659794,"The Acetate Consortium initiative (renewal, BMGF)","The Acetate Consortium initiative (renewal, NNF)",NaN,NaN,12745820,12745820.0,2025-09-22,2025-09-22


In [18]:
export_for_review(
    partial_matches_df, id_cols=['gt_row_id', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_partial_title_match_for_review.csv',
)

Saved 6 candidate matches for review -> data_review\260727_0922_gt_partial_title_match_for_review.csv


,match_key,gt_row_id,lrd_row_id,GT index,LRD index,Similarity,GT Title,LRD Title,GT Dimensions ID,LRD Identification code,GT Total (USD),LRD Total (USD),GT Project start date,LRD Project start date,is_true_match
0,1239__1446,1239,1446,326,1199,100.000000,Understanding the mechanisms associated\n wit...,Understanding the mechanisms associated with q...,grant.14440689,grant.9440850,20734,136774.0,2023-04-01,2025-03-24,True
1,1621__1446,1621,1446,336,1199,100.000000,Understanding the\n mechanisms associated wit...,Understanding the mechanisms associated with q...,grant.13004193,grant.9440850,22329,136774.0,2021-04-01,2025-03-24,True
2,1655__1446,1655,1446,347,1199,100.000000,Understanding the\n mechanisms associated wit...,Understanding the mechanisms associated with q...,grant.9871075,grant.9440850,42390,136774.0,2020-04-01,2025-03-24,True
3,1853__1446,1853,1446,400,1199,100.000000,Understanding the mechanisms associated with q...,Understanding the mechanisms associated with q...,grant.8655803,grant.9440850,21100,136774.0,2025-04-19,2025-03-24,True
4,1949__1492,1949,1492,438,1245,100.000000,ProxiMed - Exploration and Implementation of P...,ProxiMed - Exploration and Implementation of P...,NaN,NaN,4613806,4613806.0,1905-07-14,1905-07-14,True
5,1912__1475,1912,1475,420,1228,88.659794,"The Acetate Consortium initiative (renewal, BMGF)","The Acetate Consortium initiative (renewal, NNF)",NaN,NaN,12745820,12745820.0,2025-09-22,2025-09-22,True


**Pause here.** Open the exported CSV in `data/`, review the pre-filled `is_true_match` column (defaults to `True` = confirmed match -> gap-fill + remove), flip any false positives to `False` (kept), save it as `..._reviewed.csv` in the same folder (change from '..._for_review'). Then continue.

Guidance: any grants that have a different start data, funding amount, dimensions ID, or other unique distinguishing feature should be switched to FALSE to prevent deduplication and removal.

In [19]:
confirmed_partial, rejected_partial = apply_reviewed_decisions(
    partial_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_gt_partial_title_match_reviewed.csv',
    id_cols=['gt_row_id', 'lrd_row_id'],
)
print(f"{len(confirmed_partial)} confirmed duplicates (removed, no gap-fill), {len(rejected_partial)} rejected (kept as new)")

confirmed_gt_row_ids = set(confirmed_partial['gt_row_id'])
grants_tracker_data_edited = grants_tracker_data_edited[
    ~grants_tracker_data_edited['gt_row_id'].isin(confirmed_gt_row_ids)
].reset_index(drop=True)
print(f"Removed {len(confirmed_gt_row_ids)} rows from grants_tracker_data_edited ({len(grants_tracker_data_edited)} remaining)")

1 confirmed duplicates (removed, no gap-fill), 5 rejected (kept as new)
Removed 1 rows from grants_tracker_data_edited (451 remaining)


#### d. Append remaining grants tracker rows to the last report's data

Whatever's left in `grants_tracker_data_edited` after all the matching above is genuinely new - append it.

In [20]:
append_extra_map = {
    'Dimensions.ai grant ID': 'Identification code',
}
full_append_map = {**gt_lrd_col_map, **append_extra_map}

gt_for_append = grants_tracker_data_edited.rename(columns={
    gt_col: lrd_col for gt_col, lrd_col in full_append_map.items()
})
# These rows never had a genuine last-report row of their own, so there's no real lrd_row_id
# for them - but leaving it blank means every appended row with the same Grant ID collapses onto
# the same match_key ('<Grant ID>__NA') downstream, which crashes the reviewed-file lookup in
# apply_reviewed_decisions. Namespace their own gt_row_id (stamped in the load cell above) into
# lrd_row_id instead - unique per row, and clearly traceable back to grants_tracker_data_edited.
gt_for_append['lrd_row_id'] = 'GT_' + grants_tracker_data_edited['gt_row_id'].astype(str)
gt_for_append = gt_for_append[[c for c in gt_for_append.columns if c in last_report_data_edited.columns]].copy()
gt_for_append['Database'] = 'airtable'

last_report_data_edited = pd.concat([last_report_data_edited, gt_for_append], ignore_index=True)

print(f"Appended {len(grants_tracker_data_edited)} rows from grants_tracker_data_edited")
print(f"Total rows in last_report_data_edited: {len(last_report_data_edited)}")

Appended 451 rows from grants_tracker_data_edited
Total rows in last_report_data_edited: 1878


### Dimensions title matching
#### a. Full title match - REVIEW POINT 3 (exact title)

Matches on `Title translated` in `dimensions_data` against `Title` in `last_report_data_edited` (now including the grants-tracker rows appended above).

In [21]:
lrd_dim_title_map = {}
for idx, val in last_report_data_edited['Title'].items():
    norm = normalize_title(val)
    if norm:
        lrd_dim_title_map.setdefault(norm, []).append(idx)

dim_title_match_rows = []
for dim_idx, dim_row in dimensions_data.iterrows():
    norm = normalize_title(dim_row.get('Title translated'))
    if norm and norm in lrd_dim_title_map:
        for lrd_idx in lrd_dim_title_map[norm]:
            dim_title_match_rows.append({
                'Grant ID':                dim_row['Grant ID'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'Dim index':               dim_idx,
                'LRD index':               lrd_idx,
                'Title':                   dim_row.get('Title translated'),
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'Dim Funder':              dim_row.get('Funder'),
                'LRD Funder':              last_report_data_edited.at[lrd_idx, 'Funder name'],
                'Dim Total (USD)':         dim_row.get('Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'Dim Start date':          dim_row.get('Start date'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

dim_title_matches_df = pd.DataFrame(dim_title_match_rows, columns=[
    'Grant ID', 'lrd_row_id', 'Dim index', 'LRD index', 'Title', 'LRD Identification code',
    'Dim Funder', 'LRD Funder', 'Dim Total (USD)', 'LRD Total (USD)', 'Dim Start date', 'LRD Project start date',
])  # explicit columns so a zero-match run still has the right shape (not 0 columns)
print(f"{len(dim_title_matches_df)} (Dim, LRD) pairs found")
dim_title_matches_df

174 (Dim, LRD) pairs found


,Grant ID,lrd_row_id,Dim index,LRD index,Title,LRD Identification code,Dim Funder,LRD Funder,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date
0,grant.15199044,1506,8,1259,Cell-cultured octopus production through susta...,grant.14972651,Fundação para a Ciência e Tecnologia,The Research Council of Norway,59345.0,262375.0,2025-08-31,2025-01-01
1,grant.14948933,1574,20,1324,Alternative PROTeins from INdependent ocean/la...,2025-01581_Vinnova,VINNOVA,Vinnova,50952.0,NaN,2025-06-16,NaT
2,grant.15210753,1672,43,1421,Yeast-Based Ingredients for Sustainable Food P...,NaN,"Danish Ministry of Food, Agriculture and Fishe...",Danish Government,1015582.0,NaN,2025-01-01,NaT
3,grant.14960594,971,61,952,Sustainable proteins from seaweed,NaN,Research Foundation - Flanders,Research Foundation – Flanders (FWO),NaN,NaN,2024-10-01,2024-01-10
4,grant.14687220,GT_753,64,1714,Fish Cell Production Systems (FCPS) for Sustai...,need to check,"Ministry of Business, Innovation and Employment","""Ministry of Business, Innovation, and Employm...",5640062.0,6022648.0,2024-10-01,2024-09-12
...,...,...,...,...,...,...,...,...,...,...,...,...
169,grant.9643178,401,4093,401,Plant-based infant formula and toddler drink p...,NaN,Innovate UK,UK Research and Innovation (UKRI),126100.0,140106.0,2020-11-01,2020-01-01
170,grant.14062854,557,4104,557,Understanding structure formation during high ...,NaN,Belgian Federal Science Policy Office,Research Foundation – Flanders (FWO),NaN,NaN,2020-10-15,NaT
171,grant.14064206,537,4121,537,Exploring the potential of water-soluble cerea...,NaN,Belgian Federal Science Policy Office,Research Foundation – Flanders (FWO),NaN,NaN,2020-10-01,2020-10-01
172,grant.14064240,527,4252,527,The impact of processing steps on the techno-f...,"grant.14063479, grant.14064240",Belgian Federal Science Policy Office,Federal Science Policy Department (BELSPO),NaN,NaN,2020-03-04,2020-03-04


In [22]:
export_for_review(
    dim_title_matches_df, id_cols=['Grant ID', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_title_match_for_review.csv',
)

Saved 174 candidate matches for review -> data_review\260727_0922_dim_title_match_for_review.csv


,match_key,Grant ID,lrd_row_id,Dim index,LRD index,Title,LRD Identification code,Dim Funder,LRD Funder,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date,is_true_match
0,grant.15199044__1506,grant.15199044,1506,8,1259,Cell-cultured octopus production through susta...,grant.14972651,Fundação para a Ciência e Tecnologia,The Research Council of Norway,59345.0,262375.0,2025-08-31,2025-01-01,True
1,grant.14948933__1574,grant.14948933,1574,20,1324,Alternative PROTeins from INdependent ocean/la...,2025-01581_Vinnova,VINNOVA,Vinnova,50952.0,NaN,2025-06-16,NaT,True
2,grant.15210753__1672,grant.15210753,1672,43,1421,Yeast-Based Ingredients for Sustainable Food P...,NaN,"Danish Ministry of Food, Agriculture and Fishe...",Danish Government,1015582.0,NaN,2025-01-01,NaT,True
3,grant.14960594__971,grant.14960594,971,61,952,Sustainable proteins from seaweed,NaN,Research Foundation - Flanders,Research Foundation – Flanders (FWO),NaN,NaN,2024-10-01,2024-01-10,True
4,grant.14687220__GT_753,grant.14687220,GT_753,64,1714,Fish Cell Production Systems (FCPS) for Sustai...,need to check,"Ministry of Business, Innovation and Employment","""Ministry of Business, Innovation, and Employm...",5640062.0,6022648.0,2024-10-01,2024-09-12,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
169,grant.9643178__401,grant.9643178,401,4093,401,Plant-based infant formula and toddler drink p...,NaN,Innovate UK,UK Research and Innovation (UKRI),126100.0,140106.0,2020-11-01,2020-01-01,True
170,grant.14062854__557,grant.14062854,557,4104,557,Understanding structure formation during high ...,NaN,Belgian Federal Science Policy Office,Research Foundation – Flanders (FWO),NaN,NaN,2020-10-15,NaT,True
171,grant.14064206__537,grant.14064206,537,4121,537,Exploring the potential of water-soluble cerea...,NaN,Belgian Federal Science Policy Office,Research Foundation – Flanders (FWO),NaN,NaN,2020-10-01,2020-10-01,True
172,grant.14064240__527,grant.14064240,527,4252,527,The impact of processing steps on the techno-f...,"grant.14063479, grant.14064240",Belgian Federal Science Policy Office,Federal Science Policy Department (BELSPO),NaN,NaN,2020-03-04,2020-03-04,True


**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed match, gap-fill + remove". Save as `..._reviewed_{date}.csv`, then continue.

NOTE: You must save the CSV as UTF-8 format.

In [23]:
last_report_data_pre_dim_title = last_report_data_edited.copy()

confirmed_dim_title, rejected_dim_title = apply_reviewed_decisions(
    dim_title_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_title_match_reviewed.csv',
    id_cols=['Grant ID', 'lrd_row_id'],
)
print(f"{len(confirmed_dim_title)} confirmed, {len(rejected_dim_title)} rejected")

dim_title_changed_indices = set()
dim_title_cells_filled = 0
dim_title_ids_overwritten = 0

for _, match in confirmed_dim_title.iterrows():
    dim_idx = int(match['Dim index'])
    lrd_idx = int(match['LRD index'])
    dim_row = dimensions_data.loc[dim_idx]
    dim_title_cells_filled += gap_fill(dim_row, last_report_data_edited, lrd_idx, col_map, funding_cols_lrd, dim_title_changed_indices)

    # The Dimensions grant ID takes precedence over whatever's already in 'Identification code'
    # (often a funder-issued code, or nothing at all) - it's the only ID format later runs'
    # automated ID-matching trusts (the grant.NNNNNNN regex guard above), so a non-Dimensions
    # code sitting there just means this same grant needs re-matching by title every time.
    # gap_fill (via col_map) already handles the case where it's currently empty; here we handle
    # overwriting a differing non-empty value, preserving it in 'Legacy identification code'
    # first (unless something's already stashed there from an earlier run).
    new_id = str(dim_row['Grant ID']).strip()
    old_id = last_report_data_edited.at[lrd_idx, 'Identification code']
    if not is_empty(old_id) and str(old_id).strip() != new_id:
        if is_empty(last_report_data_edited.at[lrd_idx, 'Legacy identification code']):
            last_report_data_edited.at[lrd_idx, 'Legacy identification code'] = old_id
        last_report_data_edited.at[lrd_idx, 'Identification code'] = new_id
        dim_title_changed_indices.add(lrd_idx)
        dim_title_ids_overwritten += 1

confirmed_grant_ids = set(confirmed_dim_title['Grant ID'].astype(str))
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).isin(confirmed_grant_ids)
].reset_index(drop=True)

print(f"Filled {dim_title_cells_filled} missing values across {len(dim_title_changed_indices)} rows")
print(f"Overwrote 'Identification code' with the Dimensions grant ID on {dim_title_ids_overwritten} rows "
      f"(previous value preserved in 'Legacy identification code')")
print(f"Removed {len(confirmed_grant_ids)} confirmed matches from dimensions_data ({len(dimensions_data)} remaining)")

146 confirmed, 28 rejected
Filled 424 missing values across 136 rows
Overwrote 'Identification code' with the Dimensions grant ID on 56 rows (previous value preserved in 'Legacy identification code')
Removed 141 confirmed matches from dimensions_data (4211 remaining)


In [24]:
export_highlighted_diff(last_report_data_pre_dim_title, last_report_data_edited, dim_title_changed_indices,
                         AUDIT_DIR / f'{RUN_TIMESTAMP}_dim_title_changes_last_report_data.xlsx')

,Title,Abstract,Original title,Database,Total amount,Gov contribution,Currency,Total amount (USD),Gov contribution (USD),Total amount (EUR),...,2027,2028,2029,2030,2031,2032,2033,2034,2035,lrd_row_id
4,SEEDFOOD: Functional and palatable plant seed ...,The Foundation has awarded one of the 2021 gra...,SEEDFOOD: Functional and palatable plant seed ...,airtable,55900000,55900000,DKK,8172831.0,0.0,7267000.000,...,1038142.857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4
13,GRAPHENE MEMBRANES - FUTURE ENABLER FOR BETTER...,The aim of this project is to explore the appl...,Graphene membranes - Future enabler for better...,airtable,19980675,19980675,DKK,2912812.0,0.0,2597487.750,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,13
19,SYNAGRI: Developing synergies between the bioe...,The aim of SYNAGRI is to understand the capaci...,SYNAGRI: Developing synergies between the bioe...,airtable,11955980,11955980,NOK,1213201.0,1213201.0,1040170.260,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19
32,Blue fields for a green future -seaweed as a s...,This project recognizes the great potential of...,Blue fields for a green future -seaweed as a s...,airtable,7999955,7999955,SEK,751996.0,751996.0,727995.905,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32
33,Tying blue and green resources via 3D food pri...,A shift toward production and consumption of f...,Tying blue and green resources via 3D food pri...,airtable,7999174,7999174,SEK,751922.0,751922.0,727924.834,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1451,Plant Breeding Partnership: Public Sector Coor...,"Iowa State University and its partners, Tennes...",PLANT BREEDING PARTNERSHIP: Public sector coor...,airtable,727929,727929,USD,727929,727929.0,664297.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GT_52
1467,Porphyridium cruentum: A fast growing microalg...,Porphyridium Cruentum: A Fast Growing Microalg...,Porphyridium Cruentum: A Fast Growing Microalg...,airtable,200000,200000,USD,200000,200000.0,184845.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GT_109
1627,COVID-19: Assessing zoonotic pandemic and publ...,COVID-19; zoonotic disease; public health; ris...,COVID-19: Assessing zoonotic pandemic and publ...,airtable,18600,18600,CAD,13810,13810.0,12258.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GT_376
1671,Consumers as co-developers in cell-cultured me...,Novel foods have emerged to address pressing g...,Consumers as co-developers in cell-cultured me...,airtable,180000,180000,USD,180000,180000.0,157397.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,GT_486


#### b. Partial title match - REVIEW POINT 4 (fuzzy title)

The original prototype found far fewer partial matches here (6) than a prior manual pass reportedly found (73). The diagnostic cell below narrows down *why* - it does not resolve the discrepancy on its own; getting the original methodology to compare against is still needed for a definitive answer.

In [25]:
print(f"Candidate pool sizes at this point: dimensions_data={len(dimensions_data)}, last_report_data_edited={len(last_report_data_edited)}")

_lrd_titles = [normalize_title(v) for v in last_report_data_edited['Title'] if not is_empty(v)]
_dim_titles_translated = [normalize_title(v) for v in dimensions_data['Title translated'] if not is_empty(v)]
print(f"Populated 'Title translated' in dimensions_data: {len(_dim_titles_translated)}/{len(dimensions_data)}")

print("\nMatch counts by threshold (token_sort_ratio):")
for threshold in (85, 80, 75, 70):
    count = sum(
        1
        for dim_norm in _dim_titles_translated
        for lrd_norm in _lrd_titles
        if _rfuzz.token_sort_ratio(dim_norm, lrd_norm) >= threshold
    )
    print(f"  threshold {threshold}: {count} (Dim, LRD) pairs")

print(f"\nMatch counts by scorer (threshold={FUZZY_THRESHOLD}):")
for scorer_name, scorer in (('token_sort_ratio', _rfuzz.token_sort_ratio),
                             ('partial_ratio', _rfuzz.partial_ratio),
                             ('WRatio', _rfuzz.WRatio)):
    count = sum(
        1
        for dim_norm in _dim_titles_translated
        for lrd_norm in _lrd_titles
        if scorer(dim_norm, lrd_norm) >= FUZZY_THRESHOLD
    )
    print(f"  {scorer_name}: {count} (Dim, LRD) pairs")

Candidate pool sizes at this point: dimensions_data=4211, last_report_data_edited=1878
Populated 'Title translated' in dimensions_data: 4211/4211

Match counts by threshold (token_sort_ratio):
  threshold 85: 121 (Dim, LRD) pairs
  threshold 80: 121 (Dim, LRD) pairs
  threshold 75: 127 (Dim, LRD) pairs
  threshold 70: 166 (Dim, LRD) pairs

Match counts by scorer (threshold=85):
  token_sort_ratio: 121 (Dim, LRD) pairs
  partial_ratio: 259 (Dim, LRD) pairs
  WRatio: 2170970 (Dim, LRD) pairs


In [26]:
lrd_dim_partial_list = [
    (idx, normalize_title(val))
    for idx, val in last_report_data_edited['Title'].items()
    if not is_empty(val)
]

dim_partial_match_rows = []
for dim_idx, dim_row in dimensions_data.iterrows():
    dim_norm = normalize_title(dim_row.get('Title translated'))
    if not dim_norm:
        continue
    for lrd_idx, lrd_norm in lrd_dim_partial_list:
        score = _rfuzz.token_sort_ratio(dim_norm, lrd_norm)
        if score >= FUZZY_THRESHOLD:
            dim_partial_match_rows.append({
                'Grant ID':                dim_row['Grant ID'],
                'lrd_row_id':              last_report_data_edited.at[lrd_idx, 'lrd_row_id'],
                'Dim index':               dim_idx,
                'LRD index':               lrd_idx,
                'Similarity':              score,
                'Dim Title':               dim_row.get('Title translated'),
                'LRD Title':               last_report_data_edited.at[lrd_idx, 'Title'],
                'LRD Identification code': last_report_data_edited.at[lrd_idx, 'Identification code'],
                'Dim Funder':              dim_row.get('Funder'),
                'LRD Funder':              last_report_data_edited.at[lrd_idx, 'Funder name'],
                'Dim Total (USD)':         dim_row.get('Total amount (USD)'),
                'LRD Total (USD)':         last_report_data_edited.at[lrd_idx, 'Total amount (USD)'],
                'Dim Start date':          dim_row.get('Start date'),
                'LRD Project start date':  last_report_data_edited.at[lrd_idx, 'Project start date'],
            })

dim_partial_matches_df = pd.DataFrame(dim_partial_match_rows, columns=[
    'Grant ID', 'lrd_row_id', 'Dim index', 'LRD index', 'Similarity', 'Dim Title', 'LRD Title',
    'LRD Identification code', 'Dim Funder', 'LRD Funder', 'Dim Total (USD)', 'LRD Total (USD)',
    'Dim Start date', 'LRD Project start date',
]).sort_values('Similarity', ascending=False).reset_index(drop=True)  # explicit columns so a zero-match run still has the right shape
print(f"{len(dim_partial_matches_df)} (Dim, LRD) pairs found at threshold {FUZZY_THRESHOLD}")
dim_partial_matches_df

121 (Dim, LRD) pairs found at threshold 85


,Grant ID,lrd_row_id,Dim index,LRD index,Similarity,Dim Title,LRD Title,LRD Identification code,Dim Funder,LRD Funder,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date
0,grant.15199044,1506,8,1259,100.000000,Cell-cultured octopus production through susta...,Cell-cultured octopus production through susta...,grant.14972651,Fundação para a Ciência e Tecnologia,The Research Council of Norway,59345.0,262375.0,2025-08-31,2025-01-01 00:00:00
1,grant.15425977,GT_1244,96,1758,100.000000,Enhancing circular bioeconomy pathways of plan...,Enhancing circular bioeconomy pathways of\n p...,grant.14450867,Natural Sciences and Engineering Research Council,Natural Sciences and Engineering Research Coun...,23233.0,23696.0,2024-04-01,2023-04-01 00:00:00
2,grant.15425863,GT_1243,97,1757,100.000000,Understanding the mechanisms of pulse microstr...,Understanding the mechanisms of pulse\n micro...,grant.14440710,Natural Sciences and Engineering Research Council,Natural Sciences and Engineering Research Coun...,20330.0,20734.0,2024-04-01,2023-04-01 00:00:00
3,grant.15406697,GT_1237,104,1751,100.000000,Cellular Agriculture: Directed Evolution of Gr...,Cellular Agriculture: Directed Evolution\n of...,grant.14456513,Natural Sciences and Engineering Research Council,Natural Sciences and Engineering Research Coun...,50100.0,52578.0,2024-04-01,2023-04-01 00:00:00
4,grant.14869386,872,345,853,100.000000,SAFE MEAT ANALOGUES MADE WITH ALTERNATIVE PROT...,SAFE MEAT ANALOGUES MADE WITH ALTERNATIVE PROT...,grant.14865185,State Research Agency,State Research Agency,117504.0,262854.0,2022-01-01,2022-01-01 00:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,grant.13529284,218,213,218,86.153846,KMU-innovativ -KMUi-BÖ05: MyScaleUm - Sustaina...,KMUi-BÖ05: MyScaleUm - Sustainable production ...,NaN,Federal Ministry of Education and Research,Federal Ministry of Education and Research (BMBF),498310.0,389749.0,2023-05-01,2023-05-01 00:00:00
117,grant.14960389,1405,2767,1160,86.010363,MATCHMAKING: Uncovering Optimal Soybean-Rhizob...,MATCHMAKING: Unveiling Optimal Soybean-Rhizobi...,NaN,Research Foundation - Flanders,Research Foundation Flanders,NaN,NaN,2024-10-01,2024-10-01 00:00:00
118,grant.14960389,705,2767,691,86.010363,MATCHMAKING: Uncovering Optimal Soybean-Rhizob...,MATCHMAKING: Unveiling Optimal Soybean-Rhizobi...,NaN,Research Foundation - Flanders,Research Foundation – Flanders (FWO),NaN,NaN,2024-10-01,2024-10-01 00:00:00
119,grant.14075476,840,1033,821,85.333333,"BLUEWAYSE, BLUE WAY to a Sustainable Europe",BLUE WAY TO A SUSTAINABLE EUROPE,grant.14876718,The Research Council of Norway,State Research Agency,448363.0,198802.0,2024-01-01,2024-01-01 00:00:00


In [27]:
export_for_review(
    dim_partial_matches_df, id_cols=['Grant ID', 'lrd_row_id'],
    out_path=REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_partial_title_match_for_review.csv',
)

Saved 121 candidate matches for review -> data_review\260727_0922_dim_partial_title_match_for_review.csv


,match_key,Grant ID,lrd_row_id,Dim index,LRD index,Similarity,Dim Title,LRD Title,LRD Identification code,Dim Funder,LRD Funder,Dim Total (USD),LRD Total (USD),Dim Start date,LRD Project start date,is_true_match
0,grant.15199044__1506,grant.15199044,1506,8,1259,100.000000,Cell-cultured octopus production through susta...,Cell-cultured octopus production through susta...,grant.14972651,Fundação para a Ciência e Tecnologia,The Research Council of Norway,59345.0,262375.0,2025-08-31,2025-01-01 00:00:00,True
1,grant.15425977__GT_1244,grant.15425977,GT_1244,96,1758,100.000000,Enhancing circular bioeconomy pathways of plan...,Enhancing circular bioeconomy pathways of\n p...,grant.14450867,Natural Sciences and Engineering Research Council,Natural Sciences and Engineering Research Coun...,23233.0,23696.0,2024-04-01,2023-04-01 00:00:00,True
2,grant.15425863__GT_1243,grant.15425863,GT_1243,97,1757,100.000000,Understanding the mechanisms of pulse microstr...,Understanding the mechanisms of pulse\n micro...,grant.14440710,Natural Sciences and Engineering Research Council,Natural Sciences and Engineering Research Coun...,20330.0,20734.0,2024-04-01,2023-04-01 00:00:00,True
3,grant.15406697__GT_1237,grant.15406697,GT_1237,104,1751,100.000000,Cellular Agriculture: Directed Evolution of Gr...,Cellular Agriculture: Directed Evolution\n of...,grant.14456513,Natural Sciences and Engineering Research Council,Natural Sciences and Engineering Research Coun...,50100.0,52578.0,2024-04-01,2023-04-01 00:00:00,True
4,grant.14869386__872,grant.14869386,872,345,853,100.000000,SAFE MEAT ANALOGUES MADE WITH ALTERNATIVE PROT...,SAFE MEAT ANALOGUES MADE WITH ALTERNATIVE PROT...,grant.14865185,State Research Agency,State Research Agency,117504.0,262854.0,2022-01-01,2022-01-01 00:00:00,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116,grant.13529284__218,grant.13529284,218,213,218,86.153846,KMU-innovativ -KMUi-BÖ05: MyScaleUm - Sustaina...,KMUi-BÖ05: MyScaleUm - Sustainable production ...,NaN,Federal Ministry of Education and Research,Federal Ministry of Education and Research (BMBF),498310.0,389749.0,2023-05-01,2023-05-01 00:00:00,True
117,grant.14960389__1405,grant.14960389,1405,2767,1160,86.010363,MATCHMAKING: Uncovering Optimal Soybean-Rhizob...,MATCHMAKING: Unveiling Optimal Soybean-Rhizobi...,NaN,Research Foundation - Flanders,Research Foundation Flanders,NaN,NaN,2024-10-01,2024-10-01 00:00:00,True
118,grant.14960389__705,grant.14960389,705,2767,691,86.010363,MATCHMAKING: Uncovering Optimal Soybean-Rhizob...,MATCHMAKING: Unveiling Optimal Soybean-Rhizobi...,NaN,Research Foundation - Flanders,Research Foundation – Flanders (FWO),NaN,NaN,2024-10-01,2024-10-01 00:00:00,True
119,grant.14075476__840,grant.14075476,840,1033,821,85.333333,"BLUEWAYSE, BLUE WAY to a Sustainable Europe",BLUE WAY TO A SUSTAINABLE EUROPE,grant.14876718,The Research Council of Norway,State Research Agency,448363.0,198802.0,2024-01-01,2024-01-01 00:00:00,True


**Pause here.** Review the exported CSV - default `is_true_match=True` means "confirmed duplicate, remove" (no gap-fill). Save as `..._reviewed_{date}.csv`, then continue.

In [28]:
confirmed_dim_partial, rejected_dim_partial = apply_reviewed_decisions(
    dim_partial_matches_df, REVIEW_DIR / f'{RUN_TIMESTAMP}_dim_partial_title_match_reviewed.csv',
    id_cols=['Grant ID', 'lrd_row_id'],
)
print(f"{len(confirmed_dim_partial)} confirmed duplicates (removed, no gap-fill), {len(rejected_dim_partial)} rejected (kept as new)")

# No general gap-fill for partial/fuzzy matches (too uncertain) - but the Dimensions ID itself is
# worth taking on any confirmed match, same reasoning as REVIEW POINT 3: it's the only ID format
# later runs' automated matching trusts. Fill it where 'Identification code' is empty; where a
# differing non-Dimensions code is already there, preserve it in 'Legacy identification code'
# first (unless something's already stashed there).
dim_partial_ids_overwritten = 0
for _, match in confirmed_dim_partial.iterrows():
    lrd_idx = int(match['LRD index'])
    new_id = str(match['Grant ID']).strip()
    old_id = last_report_data_edited.at[lrd_idx, 'Identification code']
    if is_empty(old_id):
        last_report_data_edited.at[lrd_idx, 'Identification code'] = new_id
        dim_partial_ids_overwritten += 1
    elif str(old_id).strip() != new_id:
        if is_empty(last_report_data_edited.at[lrd_idx, 'Legacy identification code']):
            last_report_data_edited.at[lrd_idx, 'Legacy identification code'] = old_id
        last_report_data_edited.at[lrd_idx, 'Identification code'] = new_id
        dim_partial_ids_overwritten += 1

confirmed_grant_ids = set(confirmed_dim_partial['Grant ID'].astype(str))
dimensions_data = dimensions_data[
    ~dimensions_data['Grant ID'].astype(str).isin(confirmed_grant_ids)
].reset_index(drop=True)
print(f"Set 'Identification code' to the Dimensions grant ID on {dim_partial_ids_overwritten} rows "
      f"(previous non-empty value, if any, preserved in 'Legacy identification code')")
print(f"Removed {len(confirmed_grant_ids)} rows from dimensions_data ({len(dimensions_data)} remaining)")

44 confirmed duplicates (removed, no gap-fill), 77 rejected (kept as new)
Set 'Identification code' to the Dimensions grant ID on 44 rows (previous non-empty value, if any, preserved in 'Legacy identification code')
Removed 34 rows from dimensions_data (4177 remaining)


### Final outputs

Two separate accumulating tables, deliberately kept apart because they have different schemas and different lifecycles:

- **`{DEDUP_TABLE}`** (DuckDB table, per-run) - the genuinely new/untracked grants, DSL/Dimensions schema. This is the programmatic contract S3 reads from (it's an automated script, not a notebook, so it needs a stable table to read, not an Excel file someone has to re-save). S3 then grows the permanent `funding_classified` table from these rows.
- **`funding_curated`** (DuckDB table, rebuilt fresh every run) - the gap-filled, merged last-report + Grants Tracker dataset, in the GFI tracker schema (~80 columns, same shape as `Funding{LAST_REPORT_YEAR}_inscope.xlsx`). Written via `CREATE OR REPLACE`, not accumulated - it's fully recomputed from whichever `LAST_REPORT_FILE` / `GRANTS_TRACKER_FILE` are configured each time this notebook runs, so next cycle you can simply point `LAST_REPORT_FILE` at this run's `funding_curated` export and start again from there.

Note for later: a future stage (not built yet, same boundary as the deferred manual-review/category-labelling work) will map `funding_classified` rows that clear LLM scope screening *and* human review into `funding_curated`'s schema and append them there - `funding_curated` is meant to keep growing that way over time, on top of what this notebook seeds it with.

In [29]:
con = duckdb.connect(str(DB_PATH))
con.execute(f"CREATE OR REPLACE TABLE {DEDUP_TABLE} AS SELECT * FROM dimensions_data")

last_report_data_edited['date_curated'] = RUN_DATE
con.execute("CREATE OR REPLACE TABLE funding_curated AS SELECT * FROM last_report_data_edited")
con.close()

print(f"Saved {len(dimensions_data)} new/untracked grants -> table '{DEDUP_TABLE}' (S3's input)")
print(f"Saved {len(last_report_data_edited)} in-scope rows -> table 'funding_curated'")

# Human-readable Excel mirror as well
last_report_data_edited.to_excel(AUDIT_DIR / f'{RUN_TIMESTAMP}_last_report_data_edited_{RUN_TABLE}.xlsx', index=False)
print(f"Excel copy also saved to {AUDIT_DIR}/last_report_data_edited_{RUN_TABLE}.xlsx")

Saved 4177 new/untracked grants -> table 'run_260727_0922_dedup' (S3's input)
Saved 1878 in-scope rows -> table 'funding_curated'
Excel copy also saved to data_audit/last_report_data_edited_run_260727_0922.xlsx


In [ ]:
# Sanity check: every 'in'-scope grant this pipeline has ever confirmed (funding_classified
# only grows - it's never rebuilt) should be present in the just-rebuilt funding_curated above.
# If a past year's S6-promoted grant is missing here, this rebuild has silently dropped it -
# meaning the raw last-report/grants-tracker files weren't updated with last year's promotions
# before this run. Checked by exact Identification code match rather than a 'Database' tag,
# since rows curated manually before this pipeline existed already carry 'Dimensions' as their
# Database value, making that tag unreliable as a marker of pipeline-promoted rows specifically.
# Read-only - doesn't modify either table.
check_con = duckdb.connect(str(DB_PATH))
in_scope_ids = set(check_con.sql(
    "SELECT DISTINCT \"Grant ID\" AS id FROM funding_classified WHERE scope_curated = 'in'"
).df()['id'])
curated_ids = set(check_con.sql(
    "SELECT DISTINCT \"Identification code\" AS id FROM funding_curated WHERE \"Identification code\" IS NOT NULL"
).df()['id'])
check_con.close()

missing_ids = in_scope_ids - curated_ids
if missing_ids:
    print(f"WARNING: {len(missing_ids)} 'in'-scope grant(s) from funding_classified are MISSING "
          f"from the just-rebuilt funding_curated - they may have been promoted in a previous "
          f"S6 run but never made it into this year's raw last-report/grants-tracker files.")
    print(f"Missing Grant IDs (first 20): {sorted(missing_ids)[:20]}")
else:
    print(f"OK: all {len(in_scope_ids)} 'in'-scope grants in funding_classified are present in "
          f"the rebuilt funding_curated.")